# `diff_drive` dynamics tour

This notebook is a numerical companion to `diff_drive.tex`. It does two things:

1. Lays out what the state and control vectors mean.
2. Runs four short experiments that should each produce predictable, easily-verified behavior.

If any of these tests stop matching the analytic prediction, the `core/` dynamics or its bindings have regressed.

**Prerequisites:** the release build of the pybind11 module must exist. From `AtomSim/`:

```bash
cmake --preset release && cmake --build build/release
```

The setup cell below will tell you if it's missing.

## State and control reference

**State** $x \in \mathbb{R}^5$:

| index | name    | symbol   | meaning                       | units |
|-------|---------|----------|-------------------------------|-------|
| 0     | `PX`    | $p_x$    | world-frame x-position        | m     |
| 1     | `PY`    | $p_y$    | world-frame y-position        | m     |
| 2     | `THETA` | $\theta$ | heading (CCW from +x)         | rad   |
| 3     | `V`     | $v$      | body-frame forward velocity   | m/s   |
| 4     | `OMEGA` | $\omega$ | yaw rate                      | rad/s |

**Control** $u \in \mathbb{R}^2$:

| index | symbol | meaning                              | units |
|-------|--------|--------------------------------------|-------|
| 0     | $v_L$  | commanded **left**-wheel velocity    | m/s   |
| 1     | $v_R$  | commanded **right**-wheel velocity   | m/s   |

**Continuous-time dynamics** (see `diff_drive.tex` for the derivation):

$$
\begin{aligned}
\dot p_x &= v\cos\theta \\
\dot p_y &= v\sin\theta \\
\dot\theta &= \omega \\
\dot v &= \tfrac{1}{\tau}\!\left(\tfrac{v_L+v_R}{2} - v\right) \\
\dot\omega &= \tfrac{1}{\tau}\!\left(\tfrac{v_R-v_L}{W} - \omega\right)
\end{aligned}
$$

with track width $W$ and motor time constant $\tau$. The defaults in `Params<Scalar>` are $W=0.2\,\text{m}$, $\tau=0.05\,\text{s}$.

In [ ]:
import sys
from pathlib import Path

# Walk up from the notebook's cwd until we find the AtomSim/ root (CMakePresets.json marker).
here = Path.cwd().resolve()
atomsim_root = next((p for p in [here, *here.parents] if (p / "CMakePresets.json").exists()), None)
if atomsim_root is None:
    raise RuntimeError("Could not locate AtomSim/ from the notebook's working directory.")

build_dir = atomsim_root / "build" / "release" / "dynamics" / "diff_drive" / "bindings"
if not build_dir.exists():
    raise RuntimeError(
        f"No release build at {build_dir}.\n"
        f"From {atomsim_root}, run:\n"
        f"    cmake --preset release && cmake --build build/release"
    )
sys.path.insert(0, str(build_dir))

import numpy as np
import matplotlib.pyplot as plt
import diff_drive_py as dd

PX, PY, THETA, V, OMEGA = 0, 1, 2, 3, 4

dyn = dd.DiffDriveDynamics()
print(f"track_width W = {dyn.params.track_width} m")
print(f"motor tau     = {dyn.params.tau_motor} s")


In [ ]:
def rollout(dyn, x0, u_fn, dt, T):
    """RK4 rollout from x0 for T seconds at step dt. u_fn(t) returns the 2D control."""
    n = int(round(T / dt))
    X = np.zeros((n + 1, 5))
    X[0] = x0
    t = np.arange(n + 1) * dt
    for k in range(n):
        u = np.asarray(u_fn(t[k]), dtype=float)
        X[k + 1] = dd.rk4_step(dyn, X[k], u, dt)
    return t, X


## Test 1 — Motor-lag step response

Apply $u=[1,1]$ from rest. Both wheels commanded to 1 m/s ⇒ $v_{\text{cmd}} = 1$, $\omega_{\text{cmd}} = 0$. The body velocity should follow

$$v(t) = v_{\text{cmd}}\bigl(1 - e^{-t/\tau}\bigr)$$

and yaw rate should stay identically zero. RK4 on a stable linear ODE has truncation error $\mathcal{O}(\Delta t^4)$, so the agreement should be near machine precision.

In [ ]:
x0 = np.zeros(5)
dt, T = 0.005, 0.4
t, X = rollout(dyn, x0, lambda _t: [1.0, 1.0], dt, T)

v_analytic = 1.0 * (1.0 - np.exp(-t / dyn.params.tau_motor))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t, X[:, V], label="sim $v(t)$")
ax.plot(t, v_analytic, "--", label=r"analytic $1 - e^{-t/\tau}$")
ax.set(xlabel="time [s]", ylabel="body velocity v [m/s]",
       title="Test 1: motor-lag step response")
ax.legend(); ax.grid(True); plt.show()

v_err     = float(np.max(np.abs(X[:, V] - v_analytic)))
omega_max = float(np.max(np.abs(X[:, OMEGA])))
print(f"max |v_sim - v_analytic| = {v_err:.3e}")
print(f"max |omega|              = {omega_max:.3e}")
assert v_err < 1e-6, "RK4 should match the analytic exponential closely"
assert omega_max < 1e-12, "Symmetric command must produce no yaw rate"
print("PASS")


## Test 2 — Pure rotation (no translation)

With $u=[-1,+1]$ the wheels spin equal-and-opposite, so $v_{\text{cmd}}=0$ and $\omega_{\text{cmd}} = (1-(-1))/W = 10\,\text{rad/s}$. The robot spins in place: position stays at the origin, body velocity stays zero, $\omega$ rises through the same first-order lag.

In [ ]:
x0 = np.zeros(5)
u_rot = [-1.0, 1.0]
dt, T = 0.005, 1.0
t, X = rollout(dyn, x0, lambda _t: u_rot, dt, T)

omega_ss = (u_rot[1] - u_rot[0]) / dyn.params.track_width
omega_analytic = omega_ss * (1 - np.exp(-t / dyn.params.tau_motor))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t, X[:, OMEGA], label="sim")
axes[0].plot(t, omega_analytic, "--", label="analytic")
axes[0].axhline(omega_ss, color="gray", lw=0.5)
axes[0].set(xlabel="t [s]", ylabel=r"$\omega$ [rad/s]",
            title=f"Yaw rate (steady state {omega_ss:g} rad/s)")
axes[0].legend(); axes[0].grid(True)

axes[1].plot(X[:, PX], X[:, PY], lw=2)
axes[1].scatter([0], [0], c="k", zorder=3)
axes[1].set(xlabel="px [m]", ylabel="py [m]",
            title="World position (should stay at origin)")
axes[1].set_aspect("equal"); axes[1].grid(True)
axes[1].set_xlim(-0.01, 0.01); axes[1].set_ylim(-0.01, 0.01)
plt.show()

print(f"max |px|     = {np.max(np.abs(X[:, PX])):.3e}")
print(f"max |py|     = {np.max(np.abs(X[:, PY])):.3e}")
print(f"max |v|      = {np.max(np.abs(X[:, V])):.3e}")
print(f"final omega  = {X[-1, OMEGA]:.4f}  (expect {omega_ss:.4f})")
assert np.max(np.abs(X[:, PX])) < 1e-12
assert np.max(np.abs(X[:, PY])) < 1e-12
assert np.max(np.abs(X[:, V]))  < 1e-12
print("PASS")


## Test 3 — Asymmetric command produces a circular arc

With $u=[0.5, 1.0]$: $v_{ss} = 0.75\,\text{m/s}$, $\omega_{ss} = 0.5/W = 2.5\,\text{rad/s}$, turn radius $R = v_{ss}/\omega_{ss} = 0.30\,\text{m}$. After the motor lag settles, the trajectory should trace a circle of radius $R$ centered at $(0, R)$ — we start at the origin heading $+x$ and turn left.

In [ ]:
x0 = np.zeros(5)
u_arc = [0.5, 1.0]
dt, T = 0.005, 4.0
t, X = rollout(dyn, x0, lambda _t: u_arc, dt, T)

v_ss     = (u_arc[0] + u_arc[1]) / 2
omega_ss = (u_arc[1] - u_arc[0]) / dyn.params.track_width
R = v_ss / omega_ss

phi = np.linspace(0, 2 * np.pi, 200)
circle_x = R * np.sin(phi)
circle_y = R - R * np.cos(phi)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(X[:, PX], X[:, PY], lw=2, label=f"sim trajectory  (u = {u_arc})")
ax.plot(circle_x, circle_y, "--", alpha=0.5,
        label=f"steady-state circle, R = {R:.3f} m")
ax.scatter([0], [R], c="k", marker="+", label="center")
ax.set(xlabel="px [m]", ylabel="py [m]",
       title="Test 3: constant asymmetric command -> circular arc")
ax.set_aspect("equal"); ax.legend(); ax.grid(True); plt.show()

# After ~10 tau the radius from (0, R) should be very close to R.
settle = int(0.5 / dt)
radius = np.linalg.norm(X[settle:, [PX, PY]] - np.array([0.0, R]), axis=1)
print(f"radius after settle: mean {radius.mean():.4f}, max dev {np.max(np.abs(radius - R)):.3e}")
print(f"expected radius    : {R:.4f}")
assert np.max(np.abs(radius - R)) < 5e-3
print("PASS")


## Test 4 — Linearization residual scales as $\mathcal{O}(\lVert\delta x\rVert^2)$

Pick an operating point $x^*$, $u^*$. By Taylor's theorem,

$$f(x^* + \delta x, u^*) - f(x^*, u^*) - A\,\delta x = \mathcal{O}(\lVert\delta x\rVert^2),$$

where $A = \partial f/\partial x\big|_{x^*,u^*}$. If the analytic Jacobian agrees with the nonlinear dynamics, plotting the residual norm against $\lVert\delta x\rVert$ on log-log should give a slope of 2. This is the test CLAUDE.md calls out as the load-bearing check on `linearize.hpp`.

In [ ]:
x_op = np.array([0.0, 0.0, 0.0, 0.5, 0.0])
u_op = np.array([0.5, 0.5])

J = dd.linearize(dyn, x_op, u_op)
A = np.asarray(J.A)
f_op = np.asarray(dyn.continuous(x_op, u_op))

rng = np.random.default_rng(0)
dirs = rng.standard_normal((30, 5))
dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)

scales = np.logspace(-4, -1, 16)
residuals = np.empty_like(scales)
for i, s in enumerate(scales):
    rs = []
    for d in dirs:
        dx = s * d
        f_pert = np.asarray(dyn.continuous(x_op + dx, u_op))
        rs.append(np.linalg.norm(f_pert - f_op - A @ dx))
    residuals[i] = np.mean(rs)

# Fit slope on the middle of the log-log curve (avoid float noise at the smallest scales).
fit_slice = slice(2, -2)
slope, intercept = np.polyfit(np.log(scales[fit_slice]), np.log(residuals[fit_slice]), 1)

# Reference line proportional to dx^2, anchored to a mid-range data point.
anchor = len(scales) // 2
ref_coef = residuals[anchor] / scales[anchor] ** 2
ref_line = ref_coef * scales ** 2

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(scales, residuals, "o-",
          label=r"$\|f(x^*{+}\delta x,u^*) - f(x^*,u^*) - A\,\delta x\|$")
ax.loglog(scales, ref_line, "--", alpha=0.5,
          label=r"$\propto \|\delta x\|^2$ reference")
ax.set(xlabel=r"$\|\delta x\|$", ylabel="residual norm",
       title=f"Test 4: linearization residual  (fitted slope = {slope:.3f})")
ax.legend(); ax.grid(True, which="both"); plt.show()

print(f"fitted log-log slope: {slope:.3f}  (expect 2.0)")
assert 1.9 < slope < 2.1, "Linearization residual must scale quadratically"
print("PASS: analytic Jacobian agrees with nonlinear dynamics to first order")


## Summary

If all four cells printed `PASS`, then:

- **Test 1** — motor lag and zero coupling under symmetric input.
- **Test 2** — equal-and-opposite wheel commands produce pure rotation.
- **Test 3** — asymmetric commands produce the expected steady-state arc radius $R = v_{ss}/\omega_{ss}$.
- **Test 4** — analytic Jacobian in `linearize.hpp` matches the nonlinear dynamics to first order.

Re-run this notebook whenever `core/` changes.